# selftest.py —— 基线正确性的 5 道验证

注水是凸问题的**闭式全局最优解**，但"公式对"不等于"代码对"：二分停止判据、dtype、边界 clip 任何一个写错，解可能看起来合理却违反约束。selftest 用 **5 类独立检查**把"正确性"固化成可回归的测试。

| # | 检查对象 | 验证什么 | 容差 |
|---|---|---|---|
| 1 | uniform | 能量精确用满、功率边界 | $10^{-12}$ |
| 2 | waterfilling 可行性 | 预算不超额、$0\le p\le p_{\max}$ | $10^{-10}$ |
| 3 | KKT 最优性 | 内点边际速率齐平、贴限点更高 | $10^{-3}$ |
| 4 | 闭式 vs cvxpy | 两种独立求解器对拍 | $10^{-4}$ |
| 5 | 最优性不等式 | 逐轨迹 注水 ≥ 均匀 | — |

命令行入口：`python -m baselines.selftest [--traces N]`，全部通过时 exit code 0。

## 检查 1：均匀分配的可行性

均匀解 $p_t \equiv P_{\mathrm{avg}}$ 必须满足：

$$
\sum_t p_t \Delta t = E = P_{\mathrm{avg}}\, T\, \Delta t,\qquad 0\le p_t\le p_{\max}
$$

代码检查：

```python
abs(e_u - E) < 1e-12            # 能量精确相等（float64 恒值，无求和误差问题）
p_u.max() <= p_max + 1e-15      # 不超上界
(p_u >= 0).all()                # 非负
```

容差 $10^{-12}$ 极严——均匀是常数数组，`sum()` 只是 $T\times P_{\mathrm{avg}}$，float64 下误差应在机器精度量级。这里失败说明 config 参数本身矛盾（如 $P_{\mathrm{avg}} > p_{\max}$）。

## 检查 2：注水解的可行性（硬约束）

注水解必须落在**可行集**内：

$$
\sum_t p_t^{*} \Delta t \le E,\qquad 0\le p_t^{*}\le p_{\max}
$$

代码检查：

```python
e_w <= E * (1 + 1e-10)          # 预算：只允许"少用"，不允许"超额"（单向容差）
p_w.max() <= p_max + 1e-12      # 上界
(p_w >= 0).all()                # 下界（clip 保证，双保险）
```

**注意预算检查是单向的**：$\le E(1+10^{-10})$。物理上能量预算是硬约束（电池不能超发），少用能量合法（只是次优），超额则不可行。这正是闭式二分最后用 `hi`（保证 $\sum p\le$ target）而不是 `mid` 的原因。

这个检查就是当年 float32 陷阱的"捕手"——修复前相对超额 $10^{-7}$，会被 $10^{-10}$ 容差稳定抓到。

## 检查 3：KKT 最优性（"水面齐平"）

可行性只证明"解合法"，不证明"解最优"。KKT 一阶条件才是最优性证据。对香农速率，**边际速率**（每增加单位功率的边际收益）为：

$$
\frac{\partial r_t}{\partial p_t} = B\log_2(e)\cdot\frac{g_t}{N + p_t g_t}
$$

KKT 条件要求（$\nu_t,\eta_t$ 为上下界乘子）：

$$
\frac{\partial r_t}{\partial p_t} = \lambda + \nu_t - \eta_t
$$

由此推出三类时隙的边际速率关系：

| 时隙类型 | 乘子 | 边际速率 |
|---|---|---|
| **内点** $0 < p_t < p_{\max}$ | $\nu_t=\eta_t=0$ | $=\lambda$（**全部相等**）|
| **贴上界** $p_t=p_{\max}$ | $\nu_t>0$ | $\ge\lambda$（更高）|
| **贴下界** $p_t=0$ | $\eta_t>0$ | $\le\lambda$（更低）|

代码实现：

```python
marg = B*log2(e)*g/(N + p_w*g)                          # 每时隙边际速率
inner   = (p_w > 1e-9) & (p_w < p_max - 1e-9)           # 内点掩码
clipped = p_w >= p_max - 1e-9                           # 贴上界掩码
spread = (marg[inner].max()-marg[inner].min())/marg[inner].mean()
assert spread < 1e-3                                    # 内点必须"齐平"
assert marg[clipped].min() >= marg[inner].max() - 1e-6  # 贴限点必须更高
```

这是整个 selftest 中**最有理论分量**的检查：它直接验证了二分求出的水位 $\mu$ 确实满足 KKT，而不仅仅是凑够了能量。

## 检查 4：闭式解 vs cvxpy 独立对拍

两种实现的**算法路径完全独立**：

| | 闭式二分 | cvxpy |
|---|---|---|
| 原理 | KKT 解析形式 + 标量二分 | DCP 建模 + 内点法（SCS/CLARABEL）|
| 依赖 | 仅 numpy | 外部求解器 |
| 可能犯的错 | 二分方向反、区间错、clip 错 | 归一化错、约束方向错 |

若两条独立路径得到的速率一致，则**两边同时出错且错误恰好抵消的概率极低**——这是交叉验证的逻辑。

```python
if abs(r_w - r_c) / r_c > 1e-4:    # 速率相对差
    FAIL
```

实测偏差 $\sim 10^{-7}$（cvxpy 的 SCS 是一阶方法，默认精度 $10^{-5}\sim10^{-8}$，所以容差放宽到 $10^{-4}$）。

> 注意：对拍用的是**速率**而不是功率向量。凸问题最优值唯一，但最优解在边界（$p=0$ 的时隙集合）附近可能不唯一/不连续，比较功率向量会产生假阳性；比较目标值（速率）才稳健。

## 检查 5：逐轨迹 注水 ≥ 均匀

凸优化的基本保证：**全局最优值不低于同一可行集内的任意可行解**。均匀分配是可行解（检查 1 已确认），所以必有：

$$
R_{\mathrm{wf}}(g)\ge R_{\mathrm{uniform}}(g),\qquad \forall\, g
$$

```python
if r_w < r_u - 1e-6:    # 逐轨迹检查，不取均值
    FAIL
```

- **逐轨迹**而非只比均值：均值上注水赢可能掩盖个别轨迹的异常（比如二分在某个极端 $g$ 上失败）
- 容差 $10^{-6}$ bits（绝对值）：吸收纯浮点噪声
- 这也是论文核心不等式 **均匀 < 学习策略 ≤ 注水** 的右半边基石——先确认"上界确实在上面"

### 5 类检查的逻辑层次

```
检查 1/2：解合法吗？（可行性）           ── 失败 = 约束被违反
检查 3  ：解最优吗？（一阶必要条件）      ── 失败 = 二分/公式有误
检查 4  ：和独立实现一致吗？（交叉验证）   ── 失败 = 至少一方实现错
检查 5  ：上界关系成立吗？（实验前提）    ── 失败 = 整个对比框架失效
```

In [1]:
# 演示：直接调用 run_selftest
import sys, os
# 自动定位项目根（无论 cwd 是项目根还是 baselines/）
import config
ROOT = os.path.dirname(os.path.abspath(config.__file__))  # config.py 就在项目根
sys.path.insert(0, ROOT)
os.chdir(ROOT)                      # selftest 内部用相对路径 traces/...
from baselines.selftest import run_selftest

ok = run_selftest(n_traces=20)
print("=== 自检全部通过 ===" if ok else "=== 存在失败项 ===")


v= 30 km/h: uniform=15.16 Mb  WF=16.50 Mb  增益=8.9%  [T=1000, 抽查 20 条]


v=120 km/h: uniform=14.22 Mb  WF=15.60 Mb  增益=9.7%  [T=1000, 抽查 20 条]


v=350 km/h: uniform=7.76 Mb  WF=9.50 Mb  增益=22.4%  [T=1000, 抽查 20 条]
=== 自检全部通过 ===


In [2]:
# 深入：单条轨迹上手动验证 KKT"水面齐平"
import numpy as np
from config import SceneConfig
from channel.trace_io import load_traces
from baselines import waterfilling

cfg = SceneConfig()
g, _, _ = load_traces(os.path.join(ROOT, 'traces', 'sionna_speed120_test.npz'))
g0 = g[0]
p_w = waterfilling(g0, cfg.avg_power, cfg.p_max, cfg.noise_power)

# 边际速率
marg = cfg.bandwidth_hz * np.log2(np.e) * g0 / (cfg.noise_power + p_w * g0)
inner   = (p_w > 1e-9) & (p_w < cfg.p_max - 1e-9)
clipped = p_w >= cfg.p_max - 1e-9
zero    = p_w <= 1e-9

print(f"时隙分类: 贴0={zero.sum():3d}  内点={inner.sum():3d}  贴p_max={clipped.sum():3d}  (共 {len(p_w)})")
print(f"内点边际速率: 均值={marg[inner].mean():.3e}  相对散布={(marg[inner].max()-marg[inner].min())/marg[inner].mean():.2e}")
if clipped.any():
    print(f"贴上界边际速率: 最小={marg[clipped].min():.3e}  (应 >= 内点 {marg[inner].max():.3e})")
print(f"贴0时隙边际速率: 最大={marg[zero].max():.3e}  (应 <= 内点 {marg[inner].min():.3e})")


时隙分类: 贴0=329  内点=671  贴p_max=  0  (共 1000)
内点边际速率: 均值=1.088e+08  相对散布=5.48e-16
贴0时隙边际速率: 最大=1.081e+08  (应 <= 内点 1.088e+08)
